In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet, LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, r2_score, classification_report
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

ModuleNotFoundError: No module named 'statsmodels'

In [ ]:
# Task 1a - Load Data
languages_df = pd.read_csv('languages.tsv', sep='\t')
forms_df = pd.read_csv('forms.tsv', sep='\t')
print(f"Languages: {languages_df.shape}, Forms: {forms_df.shape}")
print("\nLanguages columns:", languages_df.columns.tolist())
print("Forms columns:", forms_df.columns.tolist())
print("\nFirst few rows of languages_df:")
print(languages_df.head())
print("\nFirst few rows of forms_df:")
print(forms_df.head())

In [ ]:
# Task 1b - Aggregate and Merge Data
# Aggregate forms data by language
forms_agg = forms_df.groupby('isocode').agg({
    'wordLength': 'mean',
    'longestClusterLength': 'mean',
    'vowConsRatio': 'mean'
}).round(3)

# Rename columns as specified
forms_agg.columns = ['avgLength', 'avgCluster', 'avgVowRatio']
forms_agg.reset_index(inplace=True)

# Merge with languages dataframe
main_df = languages_df.merge(forms_agg, on='isocode', how='left')
print(f"Main dataframe shape: {main_df.shape}")
print("\nMain dataframe columns:", main_df.columns.tolist())
print("\nFirst few rows of merged data:")
print(main_df.head())


In [ ]:
# Task 1c - Find Min/Max Values
numerical_vars = ['inventorySize', 'avgLength', 'avgCluster', 'avgVowRatio']
print("Extreme values for each numerical variable:")
print("=" * 50)

for var in numerical_vars:
    if var in main_df.columns:
        min_idx = main_df[var].idxmin()
        max_idx = main_df[var].idxmax()
        min_val = main_df.loc[min_idx, var]
        max_val = main_df.loc[max_idx, var]
        min_lang = main_df.loc[min_idx, 'name']
        max_lang = main_df.loc[max_idx, 'name']
        
        print(f"{var}:")
        print(f"  Min: {min_lang} ({min_val:.3f})")
        print(f"  Max: {max_lang} ({max_val:.3f})")
        print()


In [ ]:
# Task 1d - Create Pairplot
plt.figure(figsize=(12, 10))
if all(col in main_df.columns for col in numerical_vars):
    plot_data = main_df[numerical_vars + ['family']].copy()
    g = sns.pairplot(plot_data, hue='family', diag_kind='hist', 
                     plot_kws={'alpha': 0.7}, diag_kws={'alpha': 0.7})
    g.fig.suptitle('Pairwise Distributions of Sound System Properties by Language Family', 
                   y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Some required columns are missing for pairplot")

# Correlation matrix
corr_matrix = main_df[numerical_vars].corr()
print("\nCorrelation matrix:")
print(corr_matrix.round(3))

# Visualize correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.3f')
plt.title('Correlation Matrix of Sound System Properties')
plt.tight_layout()
plt.show()


# Task 2a - Choose Statistical Test
Task 2: Testing whether Families Differ in Inventory Size
Research Question: Do sound inventory sizes differ between language families?
Approach: One-way ANOVA (Analysis of Variance)
Justification:
We have one continuous dependent variable (inventory size)
We have one categorical independent variable (language family) with 4 groups
We want to test if the means of inventory size differ across groups
ANOVA is appropriate for comparing means across multiple groups

In [ ]:
# Task 2b - Check ANOVA Assumptions
anova_data = main_df[['inventorySize', 'family']].dropna()
print(f"\nANOVA analysis using {anova_data.shape[0]} observations")

# Descriptive statistics by family
print("\nDescriptive statistics by family:")
family_stats = anova_data.groupby('family')['inventorySize'].describe()
print(family_stats)

# Check normality assumption (Shapiro-Wilk test for each group)
print("\nChecking normality assumption (Shapiro-Wilk test):")
for family in anova_data['family'].unique():
    family_data = anova_data[anova_data['family'] == family]['inventorySize']
    if len(family_data) >= 3:  # Shapiro-Wilk requires at least 3 observations
        stat, p_value = stats.shapiro(family_data)
        print(f"{family}: W = {stat:.4f}, p = {p_value:.4f}")
    else:
        print(f"{family}: Too few observations for normality test")

# Check homogeneity of variances (Levene's test)
family_groups = [group['inventorySize'].values for name, group in anova_data.groupby('family')]
levene_stat, levene_p = stats.levene(*family_groups)
print(f"\nLevene's test for equal variances:")
print(f"F = {levene_stat:.4f}, p = {levene_p:.4f}")

if levene_p > 0.05:
    print("✓ Homogeneity of variances assumption appears to be met")
else:
    print("✗ Homogeneity of variances assumption may be violated")


In [ ]:
# Task 2c - Perform ANOVA
print("\nPerforming One-way ANOVA:")
print("-" * 30)

# Fit ANOVA model
model = ols('inventorySize ~ C(family)', data=anova_data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("ANOVA Results:")
print(anova_table)

# Extract key statistics
f_stat = anova_table.loc['C(family)', 'F']
p_value = anova_table.loc['C(family)', 'PR(>F)']
alpha = 0.05

print(f"\nKey Results:")
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Alpha level: {alpha}")

# Interpret results
if p_value < alpha:
    print(f"\n✓ SIGNIFICANT RESULT: p = {p_value:.4f} < {alpha}")
    print("We reject the null hypothesis. There are significant differences in inventory size between language families.")
    
    # Post-hoc analysis if significant
    try:
        tukey = pairwise_tukeyhsd(endog=anova_data['inventorySize'], 
                                groups=anova_data['family'], alpha=0.05)
        print("\nPost-hoc analysis (Tukey HSD):")
        print(tukey)
    except Exception as e:
        print(f"Could not perform post-hoc analysis: {e}")
        
else:
    print(f"\n✗ NON-SIGNIFICANT RESULT: p = {p_value:.4f} >= {alpha}")
    print("We fail to reject the null hypothesis. There are no significant differences in inventory size between language families.")

# Effect size (eta-squared)
ss_between = anova_table.loc['C(family)', 'sum_sq']
ss_total = anova_table['sum_sq'].sum()
eta_squared = ss_between / ss_total
print(f"\nEffect size (η²): {eta_squared:.4f}")
